# Verification — `{{PKG}}` against `research-concept-neutral-r01.md`

Executed report, not the source of truth. Every claim lives as a test under
`tests/`; this page runs them and shows the evidence. Seed `{{SEED}}`.

In [ ]:
import ast, sys
from pathlib import Path
import numpy as np

ROOT = Path.cwd().parents[1]
sys.path.insert(0, str(ROOT / "src"))
rng = np.random.default_rng({{SEED}})

for file in sorted((ROOT / "src" / "{{PKG}}").rglob("*.py")):
    if file.name == "__init__.py":
        continue
    for node in ast.parse(file.read_text()).body:
        if any(getattr(t, "id", None) == "__provenance__" for t in getattr(node, "targets", [])):
            p = ast.literal_eval(node.value)
            print(f"{file.name:<18} {p['revision']:<26} eqs {','.join(p['equations'])}")

## Levels 1, 2 and 4-5 — the suite

In [ ]:
import pytest

code = pytest.main(["-q", str(ROOT / "tests"), "--rootdir", str(ROOT)])
assert code == 0, f"test suite failed (pytest exit code {code})"

## Level 3 — synthetic evidence

The declared constant caps the discrepancy at a quarter; the remedy would use
the whole range. Both are reported, neither is adopted here.

In [ ]:
from {{PKG}}.aggregate import bounded_map, convex_aggregate
from {{PKG}}.discrepancy import normalized_discrepancy

w = np.full(4, 0.25)
print(f"{'shift':>6} | {'declared /4':>12} | {'attainable /1':>14}")
for shift in (0.0, 2.0, 6.0, 20.0):
    a = convex_aggregate(bounded_map(np.full(4, -shift)), w)
    b = convex_aggregate(bounded_map(np.full(4, shift)), w)
    print(f"{shift:>6} | {normalized_discrepancy(a, b):>12.6f} | {abs(a - b):>14.6f}")